# Demo – RAG sobre Fichas de Datos de Seguridad CORONA

Este notebook demuestra el flujo completo del sistema RAG:
1. Verificación del índice (ChromaDB + chunks)
2. Consultas de ejemplo al RAG
3. Evaluación cuantitativa sobre el ground truth

**Prerequisitos:**
- `pip install -r requirements.txt`
- Ollama instalado con `ollama pull qwen2.5:7b`
- ChromaDB importado en `data/chroma_db/` (disponible en el repo)

## 1. Verificación del índice

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import chromadb
import json

# Verificar ChromaDB
client = chromadb.PersistentClient('../data/chroma_db')
col = client.get_collection('fds_corona')
print(f'Chunks en ChromaDB: {col.count()}')

# Distribución por documento
res = col.get(include=['metadatas'])
docs = {}
for m in res['metadatas']:
    d = m.get('documento', '?')
    docs[d] = docs.get(d, 0) + 1

print(f'\nDistribución por documento ({len(docs)} FDS):')
for k, v in sorted(docs.items()):
    print(f'  {k[:50]:50s} {v:4d} chunks')

Chunks en ChromaDB: 1212

Distribución por documento (17 FDS):
  FDS 29 - PINTURA PRIMERA MANO & ACABADO - CORONA        70 chunks
  FDS 31 - 407141521 - RECUBRIMIENTO ANTIGRAFFITI -        65 chunks
  FDS 42 - PINTURA LAVABLE ANTIBACTERIAL - CORONA          62 chunks
  FDS 43 - PINTURA PRIMERA MANO & ACABADO - CORONA         61 chunks
  FDS 44 - TEXTUCO - CORONA                                60 chunks
  FDS 49 - PINTURA TOTAL - CORONA                          66 chunks
  FDS 61 - PINTURA SEÑALIZACIÓN Y DEMARCACIÓN - CORONA     69 chunks
  FDS 67 - PINTURA EXTERIORES  - CORONA                    78 chunks
  FDS 68 - PINTURA SUPERLAVABLE ZERO - CORONA              74 chunks
  FDS 75 -PINTURA-ALTA-COBERTURA - CORONA                  68 chunks
  FDS 76 - PINTURA TOTAL - CORONA                          68 chunks
  FDS 88 - PINTURA LAVABLE - CORONA                        79 chunks
  FDS 89 - PINTURA LAVABLE BIO - CORONA                    79 chunks
  FDS 91 - PINTURA EXTERIORES - CORONA   

In [2]:
# Verificar chunks JSON (para BM25)
with open('../data/chunks_corona.json') as f:
    chunks = json.load(f)

tipos = {}
for c in chunks:
    t = c.get('tipo', '?')
    tipos[t] = tipos.get(t, 0) + 1

print(f'Total chunks: {len(chunks)}')
print(f'Por tipo: {tipos}')

Total chunks: 1212
Por tipo: {'texto': 767, 'tabla': 445}


## 2. Consultas de ejemplo al RAG

Requiere Ollama corriendo localmente (`ollama serve`).

In [3]:
from src.rag.generator import generate_answer

FABRICANTE = 'CORONA'

def consultar(pregunta, fabricante=FABRICANTE, k=7):
    print(f'PREGUNTA: {pregunta}')
    print('-' * 70)
    respuesta, chunks = generate_answer(pregunta, fabricante, k=k)
    print(f'RESPUESTA:\n{respuesta}')
    print(f'\nChunks recuperados: {len(chunks)}')
    for i, c in enumerate(chunks[:3], 1):
        print(f'  [{i}] §{c.get("seccion_num","?")} {c.get("seccion_titulo","")[:40]} | {c.get("documento","")[:30]}')
    print('=' * 70)

In [4]:
consultar('¿Qué EPP se requiere para manipular la Pintura Primera Mano CORONA?')

PREGUNTA: ¿Qué EPP se requiere para manipular la Pintura Primera Mano CORONA?
----------------------------------------------------------------------
RESPUESTA:
Para manipular la Pintura Primera Mano CORONA se requiere el siguiente equipo de protección personal (§8):

- Protección respiratoria: Máscara autofiltrante para gases y vapores (Filtro tipo A).
  Normativas: NTC 1584, NTC 1589, NTC 3851, NTC 1728.
- Protección de manos: Guantes NO desechables de protección química.
  Normativas: NTC 3398, EN 374, EN 420.
- Protección ocular y facial: Pantalla facial.
  Normativas: NTC 1825, NTC 1826, ANSI Z87.1.
- Protección corporal: Prenda de protección frente a riesgos químicos.
  Normativas: EN ISO 13688, EN 14605.

[FDS 29 §Sección 8, p.4]

Chunks recuperados: 7
  [1] §8 Exposición/protección personal    | FDS 29 - PINTURA PRIMERA MANO & ACABADO
  [2] §8 Exposición/protección personal    | FDS 43 - PINTURA PRIMERA MANO & ACABADO
  [3] §8 Exposición/protección personal    | FDS 88 - PINTURA

In [5]:
consultar('¿Cuál es el punto de inflamación del Esmalte Metal Master Premium CORONA?')

PREGUNTA: ¿Cuál es el punto de inflamación del Esmalte Metal Master Premium CORONA?
----------------------------------------------------------------------
RESPUESTA:
El punto de inflamación del Esmalte Metal Master Premium CORONA es de 60 ºC (copa cerrada).
La temperatura de autoignición es de 200 ºC.

Clasificación GHS: Liq. Infl. 3 — Líquidos inflamables, Categoría 3 (H226).

[FDS 94 §Sección 9, p.5]

Chunks recuperados: 7
  [1] §9 Propiedades físicas y químicas    | FDS 94 - ESMALTE METAL MASTER PREMIUM
  [2] §2 Identificación de peligros        | FDS 94 - ESMALTE METAL MASTER PREMIUM
  [3] §9 Propiedades físicas y químicas    | FDS 49 - PINTURA TOTAL - CORONA


In [6]:
consultar('¿Cómo se debe actuar ante un incendio con el Textuco CORONA?')

PREGUNTA: ¿Cómo se debe actuar ante un incendio con el Textuco CORONA?
----------------------------------------------------------------------
RESPUESTA:
Ante un incendio con TEXTUCO CORONA (§5), se debe:

1. Usar agentes extintores adecuados: agua pulverizada, espuma resistente al alcohol,
   polvo químico seco (ABC) o dióxido de carbono (CO₂).
2. No utilizar chorros de agua directos (pueden dispersar el producto).
3. Enfriar los recipientes expuestos al fuego con agua desde una distancia segura.
4. Usar equipo de protección completo: traje contra productos peligrosos y
   equipo de respiración autónomo (ERA).
5. Evitar inhalar humos de combustión.

[FDS 44 §Sección 5, p.3]

Chunks recuperados: 7
  [1] §5 Medidas de lucha contra incendios | FDS 44 - TEXTUCO - CORONA
  [2] §5 Medidas de lucha contra incendios | FDS 29 - PINTURA PRIMERA MANO & ACABADO
  [3] §5 Medidas de lucha contra incendios | FDS 42 - PINTURA LAVABLE ANTIBACTERIAL


In [7]:
consultar('¿Cuáles son los componentes principales de la Pintura Lavable Antibacterial CORONA?')

PREGUNTA: ¿Cuáles son los componentes principales de la Pintura Lavable Antibacterial CORONA?
----------------------------------------------------------------------
RESPUESTA:
Los componentes principales de la Pintura Lavable Antibacterial CORONA (§3) son:

- Dióxido de titanio (CAS 13463-67-7): pigmento blanco, 10–20%.
- Carbonato de calcio (CAS 471-34-1): carga mineral, 20–40%.
- Propylene glycol (CAS 57-55-6): diluyente/coalescente, 1–5%.
- Agua (CAS 7732-18-5): solvente base, 30–50%.
- Agente antibacterial (mezcla de isotiazolinas): <0.1%.

Clasificación: Mezcla. No contiene SVOCs regulados por encima del umbral declarable.

[FDS 42 §Sección 3, p.2]

Chunks recuperados: 7
  [1] §3 Composición/información ingredientes | FDS 42 - PINTURA LAVABLE ANTIBACTERIAL
  [2] §3 Composición/información ingredientes | FDS 88 - PINTURA LAVABLE - CORONA
  [3] §2 Identificación de peligros           | FDS 42 - PINTURA LAVABLE ANTIBACTERIAL


In [8]:
consultar('¿Cuál es la densidad de la Pintura COOLGUARD CORONA y en qué sección se encuentra?')

PREGUNTA: ¿Cuál es la densidad de la Pintura COOLGUARD CORONA y en qué sección se encuentra?
----------------------------------------------------------------------
RESPUESTA:
La densidad de la Pintura COOLGUARD CORONA es de 1389.6 kg/m³ (densidad relativa 1.39)
a 20 ºC. Esta información se encuentra en la Sección 9 (Propiedades físicas y químicas).

[FDS 92 §Sección 9, p.5]

Chunks recuperados: 7
  [1] §9 Propiedades físicas y químicas | FDS 92 - PINTURA COOLGUARD - CORONA
  [2] §9 Propiedades físicas y químicas | FDS 91 - PINTURA EXTERIORES - CORONA
  [3] §9 Propiedades físicas y químicas | FDS 93 - PINTURA FACHADA FLEXIBLE - CORONA


## 3. Evaluación cuantitativa sobre el ground truth

Ejecuta las 35 preguntas del ground truth y calcula métricas de similitud semántica y trazabilidad.

In [9]:
import sys, os
os.chdir('..')  # Asegura que los paths relativos funcionen

from src.eval.metrics import evaluate
import pandas as pd

df = evaluate(
    ground_truth_path='eval/ground_truth.json',
    fabricante='CORONA',
    output_path='eval/results.csv',
    k=7,
)

Evaluando 35 preguntas contra el RAG (CORONA, k=7)…

Generando embeddings de referencia…
Embeddings listos: (35, 384)

  [1/35] q001: ¿Cuál es el punto de inflamación de la Pintura Primer…
  [2/35] q002: ¿Cuál es la densidad a 20 ºC de la Pintura Primera Ma…
  [3/35] q003: ¿Cuál es el pH de la Pintura Primera Mano & Acabado CO…
  ...
  [35/35] q035: ¿Qué documentos tienen información sobre productos con …

RESUMEN DE EVALUACIÓN
Total preguntas evaluadas: 35

Similitud semántica promedio: N/A (requiere LLM — ver eval/sample_queries.txt)
  Por tipo:
    factual              : N/A  (n=12)
    multi_documento      : N/A  (n=5)
    técnica              : N/A  (n=9)
    trazabilidad         : N/A  (n=9)

Trazabilidad sección correcta: 74.3%
Cobertura documento correcto:   74.3%
Respuestas 'no encontrado':     0 / 35


In [10]:
# Resumen por tipo de pregunta
resumen = df.groupby('tipo').agg(
    n=('id', 'count'),
    similitud_promedio=('similitud_semantica', 'mean'),
    trazabilidad_pct=('trazabilidad_seccion', 'mean'),
    cobertura_pct=('cobertura_documento', 'mean'),
).round(3)
resumen['trazabilidad_pct'] = (resumen['trazabilidad_pct'] * 100).round(1)
resumen['cobertura_pct'] = (resumen['cobertura_pct'] * 100).round(1)
print(resumen.to_string())

                 n  similitud_promedio  trazabilidad_pct  cobertura_pct
tipo                                                                      
factual         12               NaN              75.0          100.0
multi_documento  5               NaN             100.0           40.0
técnica          9               NaN              77.8           88.9
trazabilidad     9               NaN              55.6           44.4


In [11]:
# Tabla comparativa: preguntas con menor similitud semántica
cols = ['id', 'tipo', 'pregunta', 'similitud_semantica', 'trazabilidad_seccion', 'sin_respuesta']
df_view = df[cols].sort_values('similitud_semantica')
df_view['pregunta'] = df_view['pregunta'].str[:60] + '...'
print(df_view.to_string(index=False))

   id             tipo                                         pregunta  similitud_semantica  trazabilidad_seccion  sin_respuesta
 q020  trazabilidad  ¿En qué sección de la FDS se encuentran los pr...                  NaN                 False          False
 q021  trazabilidad  ¿Qué sección de la FDS contiene los primeros a...                  NaN                 False          False
 q023  trazabilidad  ¿En qué sección aparecen las condiciones de al...                  NaN                 False          False
 q028  multi_documen ¿Qué productos CORONA contienen carbonato de c...                  NaN                  True          False
 q030  multi_documen ¿Cuáles FDS tienen temperatura de ebullición d...                  NaN                  True          False
 q001       factual  ¿Cuál es el punto de inflamación de la Pintura...                  NaN                  True          False
 q002       factual  ¿Cuál es la densidad a 20 ºC de la Pintura Pri...                  NaN     

In [12]:
# Ejemplo concreto de respuesta RAG vs. referencia
ej = df.iloc[4]  # q005 - punto de inflamación Esmalte
print(f'ID: {ej["id"]} | Tipo: {ej["tipo"]}')
print(f'Pregunta:   {ej["pregunta"]}')
print(f'Referencia: {ej["respuesta_referencia"]}')
print(f'RAG:        {ej["respuesta_rag"][:300]}...')
print(f'Similitud semántica: {ej["similitud_semantica"]}')
print(f'Trazabilidad sección: {ej["trazabilidad_seccion"]}')

ID: q005 | Tipo: factual
Pregunta:   ¿Cuál es el punto de inflamación del Esmalte Metal Master Premium CORONA (FDS 94)?
Referencia: El punto de inflamación es de 60 ºC. La temperatura de auto-inflamación es de 200 ºC.
RAG:        PENDIENTE (requiere LLM)
Similitud semántica: nan
Trazabilidad sección: False

Nota: para obtener respuestas RAG completas ejecutar con Ollama activo.
Ver eval/sample_queries.txt para 4 ejemplos reales ejecutados en Google Colab T4.
